In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import gzip
from Bio import SeqIO
import os
import time
import psutil

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / (1024**3):.2f} GB")
print(f"Available RAM: {mem.available / (1024**3):.2f} GB")
print(f"Used RAM: {mem.used / (1024**3):.2f} GB")

Total RAM: 48.00 GB
Available RAM: 11.20 GB
Used RAM: 14.26 GB


In [11]:
# --- File Paths and Data Settings ---
FASTA_FILE_PATH = '../data/vgp/all_vgp_tes.fa.bgz'
FEATURES_FILE_PATH = '../data/vgp/features.txt'
MODEL_SAVE_PATH = "vibe_coded_model_checkpoint.pth"

# --- Model Hyperparameters ---
# IMPORTANT: All sequences will be padded/truncated to this length.
#            Choose a value that is a multiple of 64 (4*4*4).
SEQUENCE_LENGTH = 16384
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS = 12

# --- Data Split ---
# Defines the proportion of data for training, validation, and testing.
# The numbers should sum to 1.0.
DATA_SPLIT_RATIOS = {'train': 0.7, 'val': 0.15, 'test': 0.15}

# --- Setup device (use GPU if available) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [12]:
# =============================================================================
# HELPER FUNCTION: ONE-HOT ENCODING
# =============================================================================
def one_hot_encode(sequence: str):
    """Converts a DNA sequence string into a one-hot encoded NumPy array."""
    encoding_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    one_hot = np.zeros((len(sequence), 4), dtype=np.uint8)
    for i, nucleotide in enumerate(sequence.upper()):
        if nucleotide in encoding_map:
            one_hot[i, encoding_map[nucleotide]] = 1
    return one_hot

In [13]:
# =============================================================================
# PYTORCH DATASET CLASS
# =============================================================================
class TransposonDataset(Dataset):
    """
    This version is optimized for BGZF-compressed FASTA files.
    It uses Biopython's indexing for fast, memory-efficient random access.
    """
    def __init__(self, fasta_path, features_path, seq_len):
        self.seq_len = seq_len
        print("Loading labels...")
        self.labels = self._load_labels(features_path)
        
        # This creates a persistent index file (e.g., 'your_file.fa.bgz.idx')
        # for extremely fast re-loading on subsequent runs.
        index_file = f"{fasta_path}.idx"
        print(f"Indexing BGZF FASTA file (will create/use '{index_file}')...")
        
        # The key change: Let Biopython handle the BGZF file directly.
        # It knows how to read the binary format and decompress on the fly.
        self.fasta_handle = SeqIO.index_db(index_file, fasta_path, "fasta")
        
        # Get the intersection of IDs to ensure data integrity
        fasta_keys = set(self.fasta_handle.keys())
        label_keys = set(self.labels.keys())
        self.sequence_ids = sorted(list(fasta_keys.intersection(label_keys)))
        
        if not self.sequence_ids:
            raise ValueError("No matching sequence IDs found between FASTA and features file.")
        
        print(f"Found {len(self.sequence_ids)} matching sequences.")

    def _load_labels(self, path):
        """This function correctly opens the plain-text features file."""
        labels = {}
        with open(path, "rt") as f: # "rt" is correct here for the plain text labels
            for line in f:
                if not line.strip(): continue
                parts = line.strip().lstrip('>').split('\t')
                if len(parts) == 2:
                    labels[parts[0]] = 1.0 if parts[1] == 'TRUE' else 0.0
        return labels

    def __len__(self):
        return len(self.sequence_ids)

    def __getitem__(self, idx):
        seq_id = self.sequence_ids[idx]
        label = self.labels[seq_id]
        
        # Biopython handles the decompression automatically when you access the record.
        # This is the magic of using an indexed BGZF file.
        sequence = str(self.fasta_handle[seq_id].seq)
        
        # Pad or truncate the sequence
        if len(sequence) > self.seq_len:
            sequence = sequence[:self.seq_len]
        else:
            sequence = sequence + 'N' * (self.seq_len - len(sequence))
            
        # One-hot encode and format for PyTorch
        encoded_seq = one_hot_encode(sequence)
        tensor_seq = torch.from_numpy(encoded_seq).float().permute(1, 0)
        tensor_label = torch.tensor(label, dtype=torch.float32)
        
        return tensor_seq, tensor_label

In [14]:
class TransposonDataset(Dataset):
    """
    This version is optimized for BGZF files and is fully compatible with
    multiprocessing DataLoaders (num_workers > 0). It uses a lightweight
    __init__ and lazy initialization for the file handle.
    """
    def __init__(self, fasta_path, labels_dict, sequence_ids, seq_len):
        self.seq_len = seq_len
        self.fasta_path = fasta_path # Store path, not the handle
        self.fasta_handle = None    # Initialize handle as None
        
        # Store the pre-computed labels and IDs
        self.labels = labels_dict
        self.sequence_ids = sequence_ids

    def __len__(self):
        return len(self.sequence_ids)

    def __getitem__(self, idx):
        # LAZY INITIALIZATION: Open the file handle if it doesn't exist.
        # This will happen once per worker process.
        if self.fasta_handle is None:
            index_file = f"{self.fasta_path}.idx"
            self.fasta_handle = SeqIO.index_db(index_file, self.fasta_path, "fasta")

        seq_id = self.sequence_ids[idx]
        label = self.labels[seq_id]
        sequence = str(self.fasta_handle[seq_id].seq)
        
        if len(sequence) > self.seq_len:
            sequence = sequence[:self.seq_len]
        else:
            sequence = sequence + 'N' * (self.seq_len - len(sequence))
            
        encoded_seq = one_hot_encode(sequence)
        tensor_seq = torch.from_numpy(encoded_seq).float().permute(1, 0)
        tensor_label = torch.tensor(label, dtype=torch.float32)
        return tensor_seq, tensor_label

In [15]:
# =============================================================================
# NEURAL NETWORK ARCHITECTURE
# =============================================================================
class TIRClassifier(nn.Module):
    """
    A Siamese-like Convolutional Neural Network for classifying transposons.
    
    This architecture is designed to be "reverse-complement aware." It processes
    both the forward DNA strand and its reverse-complement through the same
    convolutional layers (a shared-weight "tower") and then merges the results.
    This ensures that the prediction is identical for a sequence and its
    reverse complement, which is a fundamental property of DNA.
    """
    def __init__(self, sequence_length):
        # Standard boilerplate for PyTorch models
        super(TIRClassifier, self).__init__()

        # --- Part 1: The Shared Convolutional Tower ---
        # This is the core feature extractor. It learns to identify hierarchical
        # patterns (motifs) in the DNA sequence. It is defined once and applied
        # to both the forward and reverse-complement strands.
        self.conv_tower = nn.Sequential(
            # Block 1: Learns to detect small, fundamental motifs (e.g., 11-mers)
            # Input: (batch, 4, seq_len) -> Output: (batch, 64, seq_len/4)
            nn.Conv1d(in_channels=4, out_channels=64, kernel_size=11, padding='same'),
            nn.BatchNorm1d(64),      # Stabilizes training
            nn.ReLU(),               # Non-linear activation function
            nn.MaxPool1d(kernel_size=4), # Downsamples sequence length, creates positional invariance
            nn.Dropout(p=0.2),       # Regularization to prevent overfitting

            # Block 2: Learns patterns of motifs from the previous block
            # Input: (batch, 64, seq_len/4) -> Output: (batch, 128, seq_len/16)
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=7, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
            nn.Dropout(p=0.3),

            # Block 3: Learns even more abstract, higher-level features
            # Input: (batch, 128, seq_len/16) -> Output: (batch, 256, seq_len/64)
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding='same'),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
            nn.Dropout(p=0.4)
        )

        # --- Calculate the input size for the fully-connected layers ---
        # The sequence length is downsampled by a factor of 4 three times (4*4*4=64)
        final_seq_len = sequence_length // (4 * 4 * 4)
        # The size of the flattened vector is the number of output channels from the
        # conv tower multiplied by the final sequence length.
        flattened_size = 256 * final_seq_len

        # --- Part 2: The Classifier Head ---
        # This part takes the rich feature representation from the conv tower
        # and makes the final binary classification decision.
        self.classifier = nn.Sequential(
            # Flattens the 2D feature map (channels, length) into a 1D vector
            nn.Flatten(),
            
            # First dense layer to learn combinations of the convolutional features
            nn.Linear(in_features=flattened_size, out_features=256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.5), # Higher dropout rate is common in dense layers

            # Final output layer. A single neuron produces one "logit" value.
            # A large positive logit -> high confidence of TIR (Class 1)
            # A large negative logit -> high confidence of no TIR (Class 0)
            nn.Linear(in_features=256, out_features=1)
        )

    def forward(self, x):
        """
        Defines the forward pass of the model.
        Args:
            x (torch.Tensor): The input tensor of one-hot encoded DNA sequences.
                              Shape: (batch_size, 4, sequence_length)
        """
        # --- The Siamese Logic ---
        # 1. Create the reverse-complement on the fly.
        #    Flipping dimension 1 (channels) swaps A<->T, C<->G in our encoding.
        #    Flipping dimension 2 (length) reverses the sequence.
        x_rc = torch.flip(x, dims=[1, 2])
        
        # 2. Pass both the forward and reverse-complement strands through the
        #    *exact same* convolutional tower. This enforces weight sharing.
        features_fwd = self.conv_tower(x)
        features_rc = self.conv_tower(x_rc)
        
        # 3. Merge the features. Element-wise addition is commutative (A+B = B+A),
        #    which mathematically guarantees the output will be the same regardless
        #    of which strand was the original input. This is the key to symmetry.
        merged_features = features_fwd + features_rc
        
        # 4. Pass the final, merged feature representation to the classifier head
        #    to get the final prediction score (logit).
        logits = self.classifier(merged_features)
        
        # The output of the final Linear layer is (batch_size, 1).
        # .squeeze() removes the singleton dimension to get a clean (batch_size,) output,
        # which is expected by the BCEWithLogitsLoss function.
        return logits.squeeze(dim=1)

In [16]:
# =============================================================================
# TRAINING & EVALUATION FUNCTIONS
# =============================================================================
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    Performs a single training pass (one epoch) over the entire training dataset.

    Args:
        model (nn.Module): The PyTorch model to be trained.
        dataloader (DataLoader): The DataLoader providing batches of training data.
        optimizer (torch.optim.Optimizer): The optimizer to update the model's weights.
        loss_fn: The loss function to calculate the difference between predictions and labels.
        device (str): The device ('cpu' or 'cuda') to perform computations on.

    Returns:
        tuple: A tuple containing the average loss and accuracy over the epoch.
    """
    # --- Step 1: Set the model to training mode ---
    # This is crucial. It enables layers like Dropout and BatchNorm to function
    # correctly during training (e.g., randomly dropping neurons).
    model.train()

    # --- Initialize tracking variables for the epoch ---
    total_loss = 0.0          # To accumulate the loss over all batches
    correct_predictions = 0   # To count the number of correct predictions
    total_samples = 0         # To count the total number of samples processed

    # --- Step 2: Iterate over batches of data from the DataLoader ---
    # The DataLoader provides one batch of (inputs, labels) in each iteration.
    for inputs, labels in dataloader:
        # --- Move data to the configured device (GPU or CPU) ---
        inputs, labels = inputs.to(device), labels.to(device)

        # --- Step 3: Perform the forward pass and calculate loss ---
        # Zero out gradients from the previous iteration. If we don't, they will
        # accumulate, which is incorrect for most standard training loops.
        optimizer.zero_grad()
        
        # Forward pass: Feed the input tensor through the model to get raw output scores (logits).
        logits = model(inputs)
        
        # Calculate the loss for this batch. The loss function compares the model's
        # predictions (logits) with the true labels.
        loss = loss_fn(logits, labels)

        # --- Step 4: Perform the backward pass and update weights ---
        # Backward pass (Backpropagation): Compute the gradients of the loss with
        # respect to every learnable parameter in the model.
        loss.backward()
        
        # Update weights: The optimizer takes a step, adjusting the model's
        # parameters in the opposite direction of their gradients to minimize the loss.
        optimizer.step()

        # --- Step 5: Record loss and accuracy for this batch ---
        # loss.item() gets the scalar loss value for the batch. We multiply by the
        # number of items in the batch to get the total loss for this batch.
        total_loss += loss.item() * inputs.size(0)
        
        # To calculate accuracy, we first convert the raw logits to probabilities
        # (using sigmoid) and then to a binary prediction (0 or 1) by thresholding at 0.5.
        preds = (torch.sigmoid(logits) > 0.5).float()
        
        # Compare predictions to the true labels and count how many were correct.
        correct_predictions += (preds == labels).sum().item()
        
        # Keep track of the total number of samples processed so far.
        total_samples += labels.size(0)
        
    # --- Step 6: Calculate average loss and accuracy for the entire epoch ---
    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    
    return avg_loss, accuracy

def evaluate(model, dataloader, loss_fn, device):
    """
    Performs a single evaluation pass on a dataset (e.g., validation or test set).

    Args:
        model (nn.Module): The PyTorch model to be evaluated.
        dataloader (DataLoader): The DataLoader providing batches of data.
        loss_fn: The loss function to calculate the difference between predictions and labels.
        device (str): The device ('cpu' or 'cuda') to perform computations on.

    Returns:
        tuple: A tuple containing the average loss and accuracy over the dataset.
    """
    # --- Step 1: Set the model to evaluation mode ---
    # This is crucial. It disables layers like Dropout (so all neurons are used)
    # and makes BatchNorm layers use their fixed, learned statistics instead of
    # calculating new ones from the batch.
    model.eval()

    # --- Initialize tracking variables ---
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # --- Step 2: Disable gradient calculations ---
    # `torch.no_grad()` is a context manager that tells PyTorch not to compute
    # or store gradients. This saves memory and speeds up computation, as we
    # are not going to call `loss.backward()` during evaluation.
    with torch.no_grad():
        # --- Step 3: Iterate over batches of data ---
        for inputs, labels in dataloader:
            # --- Move data to the configured device ---
            inputs, labels = inputs.to(device), labels.to(device)
            
            # --- Step 4: Perform the forward pass and calculate loss ---
            # Forward pass: Feed the input tensor through the model to get logits.
            logits = model(inputs)
            
            # Calculate the loss for this batch.
            loss = loss_fn(logits, labels)
            
            # --- Step 5: Record loss and accuracy for this batch ---
            # (This is identical to the training loop)
            total_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct_predictions += (preds == labels).sum().item()
            total_samples += labels.size(0)
            
    # --- Step 6: Calculate average loss and accuracy for the entire dataset ---
    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    
    return avg_loss, accuracy

In [19]:
# =============================================================================
# MAIN EXECUTION BLOCK
# =============================================================================
if __name__ == "__main__":
    # --- Percentage of data to use for this run ---
    DATA_SUBSET_PERCENT = 0.02 # Use 90% of the data

    # --- Step 1: Pre-computation (Do the heavy lifting once in the main process) ---
    print("--- Pre-computation Step ---")
    
    # 1a. Load all labels from the features file into a dictionary
    print("Loading all labels...")
    all_labels = {}
    with open(FEATURES_FILE_PATH, "rt") as f:
        for line in f:
            if not line.strip(): continue
            parts = line.strip().lstrip('>').split('\t')
            if len(parts) == 2:
                all_labels[parts[0]] = 1.0 if parts[1] == 'TRUE' else 0.0
    
    # 1b. Parse all sequence IDs from the BGZF FASTA file into a list
    print("Parsing all sequence IDs from FASTA file...")
    # Using gzip.open is correct here for a simple sequential read of IDs
    with gzip.open(FASTA_FILE_PATH, "rt") as handle:
        all_fasta_ids = [record.id for record in SeqIO.parse(handle, "fasta")]

    # 1c. Find the intersection to get a clean list of IDs that exist in both files
    fasta_keys = set(all_fasta_ids)
    label_keys = set(all_labels.keys())
    final_sequence_ids = sorted(list(fasta_keys.intersection(label_keys)))
    
    print(f"Found {len(final_sequence_ids)} matching sequences.")

    # 2. Create the full dataset instance using the pre-computed data
    # This __init__ call is now very fast and lightweight.
    full_dataset = TransposonDataset(
        fasta_path=FASTA_FILE_PATH,
        labels_dict=all_labels,
        sequence_ids=final_sequence_ids,
        seq_len=SEQUENCE_LENGTH
    )
    
    # 3. Create a smaller subset for faster training/debugging
    print(f"\n--- Subsetting Data for a Faster Run ---")
    total_available_samples = len(full_dataset)
    subset_size = int(total_available_samples * DATA_SUBSET_PERCENT)
    remainder_size = total_available_samples - subset_size
    subset_dataset, _ = random_split(full_dataset, [subset_size, remainder_size])
    
    print(f"Total available samples: {total_available_samples}")
    print(f"Subset size for this run: {len(subset_dataset)}")

    # 4. Split the SUBSET into training, validation, and testing sets
    total_subset_size = len(subset_dataset)
    train_size = int(total_subset_size * DATA_SPLIT_RATIOS['train'])
    val_size = int(total_subset_size * DATA_SPLIT_RATIOS['val'])
    test_size = total_subset_size - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(subset_dataset, [train_size, val_size, test_size])
    
    print(f"\nData subset successfully split:")
    print(f"  - Training samples:   {len(train_dataset)}")
    print(f"  - Validation samples: {len(val_dataset)}")
    print(f"  - Testing samples:    {len(test_dataset)}")
    
    # 5. Create DataLoaders with the specified settings
    # Using num_workers=1 uses a single, separate process for data loading.
    # Using pin_memory=False is a safe default, especially without multiple workers.
    print("\nCreating DataLoaders with num_workers=0 and pin_memory=False...")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    
    # 6. Initialize model, loss function, and optimizer
    model = TIRClassifier(SEQUENCE_LENGTH).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    # 7. Training loop
    print(f"\n--- Starting Training (on {DATA_SUBSET_PERCENT*100:.1f}% of data) ---")
    best_val_accuracy = 0.0
    for epoch in range(EPOCHS):
        start_time = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        
        print(f"Epoch {epoch+1:02}/{EPOCHS} | "
              f"Time: {time.time() - start_time:.2f}s | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"  -> New best model saved to '{MODEL_SAVE_PATH}' with accuracy: {best_val_accuracy:.4f}")

    print("\n--- Training Finished ---")

    # 8. Final Testing on the held-out test set
    print(f"\n--- Starting Final Testing ---")
    # Check if a model was ever saved (i.e., if training happened and validation accuracy was > 0)
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"Loading best model from '{MODEL_SAVE_PATH}'...")
        final_model = TIRClassifier(SEQUENCE_LENGTH).to(device)
        final_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
        
        test_loss, test_acc = evaluate(final_model, test_loader, loss_fn, device)
        
        print("\n--- Test Results ---")
        print(f"Final performance of best model on the held-out test set:")
        print(f"  - Test Loss: {test_loss:.4f}")
        print(f"  - Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
        print("--------------------")
    else:
        print("No model was saved during training. Skipping final testing.")

--- Pre-computation Step ---
Loading all labels...
Parsing all sequence IDs from FASTA file...
Found 135751 matching sequences.

--- Subsetting Data for a Faster Run ---
Total available samples: 135751
Subset size for this run: 2715

Data subset successfully split:
  - Training samples:   1900
  - Validation samples: 407
  - Testing samples:    408

Creating DataLoaders with num_workers=0 and pin_memory=False...

--- Starting Training (on 2.0% of data) ---
Epoch 01/12 | Time: 425.81s | Train Loss: 0.6221, Train Acc: 0.6532 | Val Loss: 0.6136, Val Acc: 0.7027
  -> New best model saved to 'vibe_coded_model_checkpoint.pth' with accuracy: 0.7027
Epoch 02/12 | Time: 429.84s | Train Loss: 0.6138, Train Acc: 0.6700 | Val Loss: 0.5824, Val Acc: 0.7346
  -> New best model saved to 'vibe_coded_model_checkpoint.pth' with accuracy: 0.7346
Epoch 03/12 | Time: 421.83s | Train Loss: 0.5738, Train Acc: 0.7084 | Val Loss: 0.6121, Val Acc: 0.7101
Epoch 04/12 | Time: 960.71s | Train Loss: 0.5638, Train A

In [ ]:
if os.path.exists(MODEL_SAVE_PATH):
    print(f"Loading best model from '{MODEL_SAVE_PATH}'...")
    final_model = TIRClassifier(SEQUENCE_LENGTH).to(device)
    final_model.load_state_dict(torch.load(MODEL_SAVE_PATH))

    test_loss, test_acc = evaluate(final_model, test_loader, loss_fn, device)

    print("\n--- Test Results ---")
    print(f"Final performance of best model on the held-out test set:")
    print(f"  - Test Loss: {test_loss:.4f}")
    print(f"  - Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print("--------------------")
else:
    print("No model was saved during training. Skipping final testing.")

Loading best model from 'best_tir_classifier_model.pth'...

--- Test Results ---
Final performance of best model on the held-out test set:
  - Test Loss: 0.4091
  - Test Accuracy: 0.8514 (85.14%)
--------------------


In [ ]:
#Code to test with num_workers > 0 using multiprocessing in loading the data as well; this needs to be a separate call 
#to a different .py file because of how Jupyter handles process spawning
!/home/ahk39/miniconda3/envs/bio-env/bin/python train_model_exploration_tir.py

In [ ]:
# =============================================================================
# DEBUGGING AND VIEWING SCRIPT
# =============================================================================
if __name__ == "__main__":
    
    # --- Step 1: Check if the data files exist ---
    print("Checking for data files...")
    if not os.path.exists(FASTA_FILE_PATH):
        print(f"ERROR: FASTA file not found at '{FASTA_FILE_PATH}'")
        exit()
    if not os.path.exists(FEATURES_FILE_PATH):
        print(f"ERROR: Features file not found at '{FEATURES_FILE_PATH}'")
        exit()
    print("Data files found.")

    # --- Step 2: Create an instance of the Dataset ---
    print("\nAttempting to create TransposonDataset instance...")
    try:
        dataset = TransposonDataset(
            fasta_path=FASTA_FILE_PATH,
            features_path=FEATURES_FILE_PATH,
            seq_len=SEQUENCE_LENGTH
        )
        print("\nDataset created successfully!")
        print(f"Total number of samples in dataset: {len(dataset)}")
    except Exception as e:
        print(f"\nAn error occurred while creating the dataset: {e}")
        #exit()

    # --- Step 3: View a few samples from the dataset ---
    print("\n--- Inspecting the first 3 samples from the Dataset ---")
    
    # Ensure we don't try to access more samples than exist
    num_samples_to_view = min(3, len(dataset))
    if num_samples_to_view == 0:
        print("Dataset is empty, cannot view samples.")
    
    for i in range(num_samples_to_view):
        print(f"\n--- Sample Index: {i} ---")
        try:
            # This is where the __getitem__ method is called
            sequence_tensor, label_tensor = dataset[i]
            
            # Print detailed information about the returned tensors
            print(f"Sequence Tensor Shape: {sequence_tensor.shape}")
            print(f"  - This should be (4, {SEQUENCE_LENGTH})")
            
            print(f"Sequence Tensor dtype: {sequence_tensor.dtype}")
            print(f"  - This should be torch.float32")
            
            print(f"\nLabel Tensor Shape: {label_tensor.shape}")
            print(f"  - This should be an empty shape, indicating a scalar: torch.Size([])")

            print(f"Label Tensor Value: {label_tensor.item()}")
            print(f"  - This should be 1.0 (for TRUE) or 0.0 (for FALSE)")
            
            # Let's look at a small snippet of the one-hot encoded sequence
            # We will view the first 15 bases
            print("\nSnippet of the one-hot encoded sequence (first 15 bases):")
            # .T transposes it to (length, channels) for easier viewing
            print(sequence_tensor[:, :15].T)
            
        except Exception as e:
            print(f"An error occurred while trying to retrieve sample {i}: {e}")

Checking for data files...
Data files found.

Attempting to create TransposonDataset instance...
Loading labels...
Indexing BGZF FASTA file (will create/use '../rds/rds/projects/20251104_TEs_Alex/all_vgp_tes.fa.bgz.idx')...
Found 135751 matching sequences.

Dataset created successfully!
Total number of samples in dataset: 135751

--- Inspecting the first 3 samples from the Dataset ---

--- Sample Index: 0 ---
Sequence Tensor Shape: torch.Size([4, 65536])
  - This should be (4, 65536)
Sequence Tensor dtype: torch.float32
  - This should be torch.float32

Label Tensor Shape: torch.Size([])
  - This should be an empty shape, indicating a scalar: torch.Size([])
Label Tensor Value: 1.0
  - This should be 1.0 (for TRUE) or 0.0 (for FALSE)

Snippet of the one-hot encoded sequence (first 15 bases):
tensor([[0., 0., 0., 1.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0.,